In [ ]:
pip install -U datasets scikit-learn pandas numpy torch transformers tqdm


In [1]:
import re
import numpy as np
import pandas as pd
from datasets import load_dataset



In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix
)



In [3]:
SEED = 42
TOX_THRESHOLD = 0.5  # toxicity>=0.5 => toxic (binary)
PRED_THRESHOLD = 0.5

# Keep it small so it runs on a laptop. Increase if you have time/compute.
N_TRAIN = 50000
N_VAL   = 10000
N_TEST  = 10000



In [4]:
ds = load_dataset("google/civil_comments")  # fields include text + toxicity score
train_ds = ds["train"].shuffle(seed=SEED).select(range(N_TRAIN))
val_ds   = ds["validation"].shuffle(seed=SEED).select(range(N_VAL))
test_ds  = ds["test"].shuffle(seed=SEED).select(range(N_TEST))



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/194M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/187M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/20.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1804874 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/97320 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/97320 [00:00<?, ? examples/s]

In [5]:
def to_xy(split):
    texts = split["text"]
    y = (np.array(split["toxicity"]) >= TOX_THRESHOLD).astype(int)
    return texts, y



In [6]:
X_train, y_train = to_xy(train_ds)
X_val,   y_val   = to_xy(val_ds)
X_test,  y_test  = to_xy(test_ds)




In [7]:
IDENTITY_PATTERNS = {
    "muslim": r"\bmuslim(s)?\b",
    "christian": r"\bchristian(s)?\b",
    "jewish": r"\b(jew|jews|jewish)\b",
    "black": r"\bblack(s)?\b",
    "white": r"\bwhite(s)?\b",
    "asian": r"\basian(s)?\b",
    "gay_lesbian": r"\b(gay|lesbian|homosexual)\b",
    "transgender": r"\btrans(gender|sexual)?\b",
    "women": r"\b(woman|women|female|girl|girls)\b",
    "men": r"\b(man|men|male|boy|boys)\b",
}
_compiled = {k: re.compile(v, flags=re.IGNORECASE) for k, v in IDENTITY_PATTERNS.items()}

def group_masks(texts):
    masks = {}
    for name, pat in _compiled.items():
        masks[name] = np.array([bool(pat.search(t)) for t in texts], dtype=bool)
    masks["any_identity_term"] = np.logical_or.reduce(list(masks.values()))
    return masks





In [8]:
def metrics_at_threshold(y_true, y_prob, thr=0.5):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    fpr = fp / (fp + tn) if (fp + tn) else np.nan
    fnr = fn / (fn + tp) if (fn + tp) else np.nan

    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "fpr": fpr,
        "fnr": fnr,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
        "n": len(y_true),
    }

def evaluate(y_true, y_prob, thr=0.5, title=""):
    out = {}
    out["roc_auc"] = roc_auc_score(y_true, y_prob)
    out["pr_auc"]  = average_precision_score(y_true, y_prob)
    out.update(metrics_at_threshold(y_true, y_prob, thr))
    if title:
        print(f"\n=== {title} ===")
    print(pd.Series(out).round(4))
    return out

def bias_audit(texts, y_true, y_prob, thr=0.5):
    masks = group_masks(texts)
    rows = []

    overall = metrics_at_threshold(y_true, y_prob, thr)
    rows.append({"group": "OVERALL", **overall})

    for g, m in masks.items():
        if g == "any_identity_term":
            continue
        if m.sum() < 200:  # skip tiny groups for stability
            continue
        met = metrics_at_threshold(y_true[m], y_prob[m], thr)
        rows.append({"group": g, **met})

    # also show "any_identity_term" as a bucket
    m_any = masks["any_identity_term"]
    if m_any.sum() >= 200:
        met_any = metrics_at_threshold(y_true[m_any], y_prob[m_any], thr)
        rows.append({"group": "any_identity_term", **met_any})

    df = pd.DataFrame(rows).sort_values("group")
    # add disparity vs overall for quick bias read
    df["fpr_gap_vs_overall"] = df["fpr"] - overall["fpr"]
    df["fnr_gap_vs_overall"] = df["fnr"] - overall["fnr"]
    return df





In [9]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),
    min_df=3,
    max_features=200000
)

Xtr = vectorizer.fit_transform(X_train)
Xva = vectorizer.transform(X_val)
Xte = vectorizer.transform(X_test)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",  # helps imbalance
    n_jobs=-1
)
clf.fit(Xtr, y_train)

val_prob  = clf.predict_proba(Xva)[:, 1]
test_prob = clf.predict_proba(Xte)[:, 1]

evaluate(y_val, val_prob, thr=PRED_THRESHOLD, title="Baseline (Validation)")
evaluate(y_test, test_prob, thr=PRED_THRESHOLD, title="Baseline (Test)")

audit_base = bias_audit(X_test, y_test, test_prob, thr=PRED_THRESHOLD)
print("\n=== Bias audit (baseline) — sorted by FPR gap vs overall ===")
print(audit_base.sort_values("fpr_gap_vs_overall", ascending=False)[
    ["group","n","fpr","fnr","f1","fpr_gap_vs_overall","fnr_gap_vs_overall"]
].round(4))





/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



=== Baseline (Validation) ===
roc_auc          0.9028
pr_auc           0.5850
precision        0.4863
recall           0.6378
f1               0.5519
fpr              0.0593
fnr              0.3622
tp             516.0000
fp             545.0000
tn            8646.0000
fn             293.0000
n            10000.0000
dtype: float64

=== Baseline (Test) ===
roc_auc          0.8982
pr_auc           0.5815
precision        0.4761
recall           0.6423
f1               0.5469
fpr              0.0621
fnr              0.3577
tp             519.0000
fp             571.0000
tn            8621.0000
fn             289.0000
n            10000.0000
dtype: float64

=== Bias audit (baseline) — sorted by FPR gap vs overall ===
               group      n     fpr     fnr      f1  fpr_gap_vs_overall  \
1              white    205  0.4233  0.1429  0.4898              0.3612   
4  any_identity_term   1063  0.2004  0.2500  0.5206              0.1383   
3                men    447  0.1624  0.2830  0.4903

In [10]:
train_masks = group_masks(X_train)
has_id = train_masks["any_identity_term"]

sample_weight = np.ones_like(y_train, dtype=float)
sample_weight[(y_train == 0) & has_id] = 2.0  # upweight non-toxic identity-containing text

clf_rw = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)
clf_rw.fit(Xtr, y_train, sample_weight=sample_weight)

test_prob_rw = clf_rw.predict_proba(Xte)[:, 1]
evaluate(y_test, test_prob_rw, thr=PRED_THRESHOLD, title="Reweighted (Test)")

audit_rw = bias_audit(X_test, y_test, test_prob_rw, thr=PRED_THRESHOLD)
print("\n=== Bias audit (reweighted) — sorted by FPR gap vs overall ===")
print(audit_rw.sort_values("fpr_gap_vs_overall", ascending=False)[
    ["group","n","fpr","fnr","f1","fpr_gap_vs_overall","fnr_gap_vs_overall"]
].round(4))





/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



=== Reweighted (Test) ===
roc_auc          0.8968
pr_auc           0.5868
precision        0.4879
recall           0.6250
f1               0.5480
fpr              0.0577
fnr              0.3750
tp             505.0000
fp             530.0000
tn            8662.0000
fn             303.0000
n            10000.0000
dtype: float64

=== Bias audit (reweighted) — sorted by FPR gap vs overall ===
               group      n     fpr     fnr      f1  fpr_gap_vs_overall  \
1              white    205  0.2883  0.2381  0.5289              0.2307   
4  any_identity_term   1063  0.1318  0.3562  0.5393              0.0741   
2              women    276  0.0897  0.3571  0.6000              0.0321   
3                men    447  0.0863  0.3962  0.5378              0.0286   
0            OVERALL  10000  0.0577  0.3750  0.5480              0.0000   

   fnr_gap_vs_overall  
1             -0.1369  
4             -0.0187  
2             -0.0179  
3              0.0212  
0              0.0000  


In [11]:
def show_examples(texts, y_true, y_prob, thr=0.5, want="fp", k=5):
    y_pred = (y_prob >= thr).astype(int)
    if want == "fp":
        idx = np.where((y_true == 0) & (y_pred == 1))[0]
        title = "FALSE POSITIVES (non-toxic predicted toxic)"
    else:
        idx = np.where((y_true == 1) & (y_pred == 0))[0]
        title = "FALSE NEGATIVES (toxic predicted non-toxic)"
    print(f"\n--- {title}: showing {min(k,len(idx))} of {len(idx)} ---")
    for i in idx[:k]:
        print(f"\nscore={y_prob[i]:.3f} | true={y_true[i]} | pred={y_pred[i]}")
        print(texts[i][:400].replace("\n"," "))

show_examples(X_test, y_test, test_prob, thr=PRED_THRESHOLD, want="fp", k=5)



--- FALSE POSITIVES (non-toxic predicted toxic): showing 5 of 571 ---

score=0.601 | true=0 | pred=1
he is 18. You're making excuses. You don't have to be 25 to know not to play with Guns. Give me a break. he'll pay for his crime,  and have to live with the fact that he took a life.. But we don't need  to cover up his actions with ridiculous notion that he's too young to not make this kind of mistake.

score=0.576 | true=0 | pred=1
The police showed a lot of restraint. If the guy had followed their orders and taken his evidence to court, this incident could have been avoided. This incident represents just how lawless and violence prone members of the crowd are. They are of a criminal element and we should not fool ourselves about the degree of risk to public safety that they represent.

score=0.801 | true=0 | pred=1
If the State Dept do not stop his Ukrainian Nazi-monster, he will arrange a chemical and radiation catastrophe in Europe.

score=0.729 | true=0 | pred=1
And I suppose the 